In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import torch
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM

from src.config import MODEL_NAME, DATASETS_DIR
from src.data.prompt.pythoncodes import build_prompt
from src.data.curriculum.base import CurriculumPipeline
from src.data.curriculum.heuristics import LengthScorer
from src.data.curriculum.entropy import PPLScorer, IFDScorer
from src.data.curriculum.clustering import LexicalClusterScorer, SemanticClusterScorer
from src.data.curriculum.llm_judge import LLMJudgeScorer
from src.data.curriculum.evaluator import CurriculumEvaluator

In [2]:
print(f"Загрузка мини-датасета (100 примеров) в {DATASETS_DIR}...")
raw_dataset = load_dataset(
    "flytech/python-codes-25k",
    split="train[:1000]",
    cache_dir=DATASETS_DIR
)
print(f"Успешно! Доступные колонки: {raw_dataset.column_names}")

Загрузка мини-датасета (100 примеров) в ./datasets/...
Успешно! Доступные колонки: ['output', 'instruction', 'input', 'text']


In [3]:
raw_dataset[0]

{'output': "```python\ntasks = []\nwhile True:\n    task = input('Enter a task or type 'done' to finish: ')\n    if task == 'done': break\n    tasks.append(task)\nprint(f'Your to-do list for today: {tasks}')\n```",
 'instruction': 'Help me set up my daily to-do list!',
 'input': 'Setting up your daily to-do list...',
 'text': "Help me set up my daily to-do list! Setting up your daily to-do list... ```python\ntasks = []\nwhile True:\n    task = input('Enter a task or type 'done' to finish: ')\n    if task == 'done': break\n    tasks.append(task)\nprint(f'Your to-do list for today: {tasks}')\n```"}

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Инициализация: {MODEL_NAME} на {DEVICE.upper()}...")

Инициализация: Qwen/Qwen3-4B-Instruct-2507 на CUDA...


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(DEVICE)
model.eval()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [6]:
N_CLUSTERS = 3
scorers = [
    # 1. Базовая эвристика
    LengthScorer(
        name="length",
        tokenizer=tokenizer,
        build_prompt_fn=build_prompt
    ),

    # 2. Информационная энтропия (Перплексия)
    PPLScorer(
        name="ppl",
        tokenizer=tokenizer,
        model=model,
        build_prompt_fn=build_prompt,
        device=DEVICE
    ),

    # 3. Информационная энтропия (IFD - Instruction-Following Difficulty)
    IFDScorer(
        name="ifd",
        tokenizer=tokenizer,
        model=model,
        build_prompt_fn=build_prompt,
        device=DEVICE
    ),

    # 4. Лексические аномалии (TF-IDF)
    LexicalClusterScorer(
        name="lexical_cluster",
        tokenizer=tokenizer,
        n_clusters=N_CLUSTERS
    ),

    # 5. Семантические аномалии (Hidden States)
    SemanticClusterScorer(
        name="semantic_cluster",
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        n_clusters=N_CLUSTERS
    ),

    # 6. LLM-Судья (Когнитивная оценка)
    LLMJudgeScorer(
        name="llm_judge",
        tokenizer=tokenizer,
        model=model,
        device=DEVICE
    )
]

In [7]:
pipeline = CurriculumPipeline(scorers=scorers, percentiles=(57, 86))
display(pipeline)

In [8]:
pipeline.fit(raw_dataset)
processed_dataset = pipeline.process_dataset(raw_dataset, batch_size=10)

new_cols = [c for c in processed_dataset.column_names if c not in raw_dataset.column_names]
new_cols

[length] Обучение...
[ppl] Обучение...
[ifd] Обучение...
[lexical_cluster] Обучение...
[lexical_cluster] Обучение TF-IDF и KMeans...
[semantic_cluster] Обучение...
[semantic_cluster] Извлечение эмбеддингов для обучения кластеров...
[llm_judge] Обучение...
[length] Оцениваем датасет по степеням...


Scoring length:   0%|          | 0/1000 [00:00<?, ? examples/s]

[ppl] Оцениваем датасет по степеням...


Scoring ppl:   0%|          | 0/1000 [00:00<?, ? examples/s]

[ifd] Оцениваем датасет по степеням...


...

Scoring llm_judge:   0%|          | 0/1000 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Расчёт перцентилей для присвоения категорий


Categorizing:   0%|          | 0/1000 [00:00<?, ? examples/s]

['length_score',
 'ppl_score',
 'ifd_score',
 'lexical_cluster_score',
 'semantic_cluster_score',
 'llm_judge_score',
 'length_category',
 'ppl_category',
 'ifd_category',
 'lexical_cluster_category',
 'semantic_cluster_category',
 'llm_judge_category']

In [9]:
SAVE_PATH = os.path.join(DATASETS_DIR, "pythoncodes-curriculum")
processed_dataset.save_to_disk(SAVE_PATH)

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
loaded_dataset = load_from_disk(SAVE_PATH)

evaluator = CurriculumEvaluator(
    processed_dataset=loaded_dataset,
    save_dir="./reports/curriculum_plots_test"
)

evaluator.generate_report()

print("\n--- Корреляция Спирмена (Таблица) ---")
scores_df = evaluator.df[evaluator.score_cols].rename(columns=lambda x: x.replace('_score', ''))
display(scores_df.corr(method='spearman').style.background_gradient(cmap='vlag', vmin=-1, vmax=1))

Конвертация датасета в DataFrame для анализа...
=== Запуск оценки датасета (Total samples: 1000) ===
Генерация графиков распределения (Distributions)...
Сохранено: ./reports/curriculum_plots_test/score_distributions.png
Генерация матрицы корреляций (Spearman)...
Сохранено: ./reports/curriculum_plots_test/correlation_matrix.png
Генерация графиков баланса классов...
Сохранено: ./reports/curriculum_plots_test/category_balance.png
=== Оценка завершена. Графики сохранены в директорию. ===

--- Корреляция Спирмена (Таблица) ---


,length,ppl,ifd,lexical_cluster,semantic_cluster,llm_judge
length,1.000000,-0.605365,0.091060,-0.099172,-0.354095,0.359018
ppl,-0.605365,1.000000,0.113736,0.228000,0.282077,-0.191609
ifd,0.091060,0.113736,1.000000,0.161000,0.028233,-0.151712
lexical_cluster,-0.099172,0.228000,0.161000,1.000000,0.183288,0.026550
semantic_cluster,-0.354095,0.282077,0.028233,0.183288,1.000000,-0.099945
llm_judge,0.359018,-0.191609,-0.151712,0.026550,-0.099945,1.000000


In [11]:
print("Оценки LLM Judge:")
print(evaluator.df['llm_judge_score'].value_counts())

Оценки LLM Judge:
llm_judge_score
2.0    516
3.0    243
4.0    169
5.0     72
Name: count, dtype: int64
